In [ ]:
import os 

os.getcwd()

In [ ]:
COMPUTATIONAL_METRICS = ['duration_seconds', 'inference_time', 'mean_leaves', 'mean_nodes', 'ntrees',
        'train_time', 'get_weights_avg_time', 'get_indexers_total_time', 'get_weights_calls', 'get_weights_total_time, get_indexers_avg_time', 'get_indexers_calls']
DETAILED_COMPUTATIONAL = ['get_weights_avg_time', 'get_indexers_total_time', 'get_weights_calls', 'get_weights_total_time, get_indexers_avg_time', 'get_indexers_calls']

In [ ]:
def filter_df(df, rule):
    for k, v in rule.items():
        if k not in df.columns:
            continue
        df = df[df[k] == v]
    return df

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def get_all_runs_data(experiment_names=None, include_artifacts=False):
    """
    Extract all runs data from MLflow experiments into a comprehensive DataFrame
    
    Args:
        experiment_names: List of experiment names or None for all experiments
        include_artifacts: Whether to include artifact URIs
    
    Returns:
        DataFrame with all runs data
    """
    client = MlflowClient()
    
    # Get experiments
    if experiment_names is None:
        experiments = client.search_experiments()
    else:
        experiments = [client.get_experiment_by_name(name) for name in experiment_names]
        experiments = [exp for exp in experiments if exp is not None]
    
    all_runs_data = []
    
    for experiment in tqdm(experiments, desc="Processing experiments"):
        experiment_id = experiment.experiment_id
        experiment_name = experiment.name
        
        print(f"Processing experiment: {experiment_name}")
        
        # Get all runs for this experiment
        runs = client.search_runs(
            experiment_ids=[experiment_id],
            max_results=10000  # Adjust if you have more runs
        )
        
        for run in tqdm(runs, desc=f"Runs in {experiment_name}", leave=False):
            run_data = {
                'run_id': run.info.run_id,
                'experiment_id': experiment_id,
                'experiment_name': experiment_name,
                'run_name': run.data.tags.get('mlflow.runName', ''),
                'status': run.info.status,
                'start_time': pd.to_datetime(run.info.start_time, unit='ms'),
                'end_time': pd.to_datetime(run.info.end_time, unit='ms') if run.info.end_time else None,
                'duration_seconds': (run.info.end_time - run.info.start_time) / 1000.0 if run.info.end_time and run.info.start_time else None,
            }
            
            # Add parameters
            for key, value in run.data.params.items():
                run_data[f'param_{key}'] = value
            
            # Add metrics
            for key, value in run.data.metrics.items():
                run_data[f'metric_{key}'] = value
            
            # Add tags
            for key, value in run.data.tags.items():
                if key not in ['mlflow.runName', 'mlflow.user']:
                    run_data[f'tag_{key}'] = value
            
            # Add artifact location if requested
            if include_artifacts:
                run_data['artifact_uri'] = run.info.artifact_uri
            
            all_runs_data.append(run_data)
    
    return pd.DataFrame(all_runs_data)

def get_baselines(path, version):
    start_dir = os.getcwd()
    os.chdir(path)
    df = get_all_runs_data(['baselines_' + str(version)])
    os.chdir(start_dir)
    return df 


In [ ]:
# bsln = get_baselines('/home/leostre/Рабочий стол/py-boost/5.2', '5.2')

In [ ]:
EXCLUDE = {'experiment_id', 'experiment_name', 'status', 'start_time', 'end_time', 'run_name', 'tag_mlflow.source.name', 'tag_mlflow.source.git.commit',
       'tag_mlflow.source.type', 'param_error', 'tag_status',
       'param_total_runs', 'param_successful_runs', 'mean_f1', 'mean_accuracy', 'param_n_splits', 'param_n_successful_folds'}

def filter_data(data):
    after_exclusion_by_name = [
        col for col in data.columns if col not in EXCLUDE
    ]
    print(after_exclusion_by_name)
    statistics = ('mean', 'max', 'min', 'std', 'median')
    after_exclusion_agg = [
        col for col in after_exclusion_by_name if 'leaves' in col or 'nodes' in col or
        'tree' in col or
        not any(statistic in col for statistic in statistics) and not 'metric_fold' in col or col in ('param_stabilization_threshold', 'param_smoothing_alpha')
    ]
    filtered_data = data[after_exclusion_agg 
                        #  + ['metric_std_num_trees', 'metric_mean_num_trees',]
                         ]
    filtered_pivot = filtered_data.rename(columns={col: col.removeprefix('param_') for col in filtered_data.columns})
    return filtered_pivot

def melt_metrics(df):
    # Identify metric columns
    metric_cols = [col for col in df.columns if col.startswith('metric_')]
    print(metric_cols)
    
    # Identify ID columns (all non-metric columns)
    id_cols = [col for col in df.columns if not col.startswith('metric_')]
    print(id_cols)
    
    # Melt using pandas melt (more control)
    melted_df = df.melt(
        id_vars=id_cols,
        value_vars=metric_cols,
        var_name='metric_fold',
        value_name='value'
    )
    
    # Extract metric name and fold number using regex pattern
    pattern = r'metric_(.+?)_fold_(\d+)$'
    extracted = melted_df['metric_fold'].str.extract(pattern)
    
    # Create new columns
    melted_df['metric'] = extracted[0]
    melted_df['fold'] = extracted[1]
    
    # For metrics without fold numbers (like 'metric_total_training_time')
    # Fill NaN metric names with the original string without 'metric_' prefix
    mask = melted_df['metric'].isna()
    melted_df.loc[mask, 'metric'] = melted_df.loc[mask, 'metric_fold'].str.replace('metric_', '')
    
    # Drop the temporary column and clean up
    melted_df = melted_df.drop('metric_fold', axis=1)
    melted_df = melted_df.reset_index(drop=True)
    
    return melted_df


In [ ]:
def flatten_index(df):
    columns = df.columns
    if isinstance(columns, pd.MultiIndex):
        columns = [c[0] if not c[1] else c[1] for c in columns] 
        columns = [c if c != 'mean' else 'value' for c in columns]
    df.columns = columns  

## Compare experiments

In [ ]:
DATASET = [
    # 'mediamill',
    # 'mnist',
    # 'cifar10',
    # 'yeast', 
        #    'age_prediction'
        #    'birds', 
        #    'genbase'
    # 'mbd',
]

EXPS = [
    'sigmoid_5.2.1',
    'hyperbolic_5.2.1',
    'baselines_5.2.1',
    'lgbm_5.2.1',
    'xgboost_5.2.1',
]
TO_INT = ['sketch_outputs']
TO_FLOAT = ['smoothing_alpha', 'stabilization_threshold', 'subsample', 'lr', ]

def process_mlruns(processed, EXPS, DATASET, path='.'):
    curdir = os.getcwd()
    os.chdir(path)
    dfs = [get_all_runs_data([exp]) for exp in EXPS] 

    version = None # '5.2'
    if version:
        dfs += [get_baselines(f'../{version}', version)]
        EXPS += [f'baselines_{version}']

    for exp, df in zip(EXPS, dfs):
        if DATASET:
            df = df[df.param_dataset.isin(DATASET)]
        df = df.rename(columns={'param_learning_rate': 'param_lr'})
        print(exp, df.shape)
        df = filter_data(df)
        mlt_df = melt_metrics(df)
        print(exp, mlt_df.shape)
        agg_mtrs = mlt_df.groupby(['dataset',
                                    *(['sketch_method', 'sketch_outputs'] if 'sketch_method' in mlt_df.columns else []), 
                                    *(['smoothing_alpha', 'stabilization_threshold'] if 'smoothing_alpha' in mlt_df.columns else []),
                                    'subsample', 'lr', 'metric', ]).agg({'value': ['mean', 'std']})#.reset_index()
        flatten_index(agg_mtrs)
        for c in TO_FLOAT:
            if not c in agg_mtrs:
                continue
            agg_mtrs[c] = agg_mtrs[c].astype(float)
        for c in TO_INT:
            if not c in agg_mtrs:
                continue
            agg_mtrs[c] = agg_mtrs[c].astype(int)
        agg_mtrs['experiment'] = exp
        processed[exp] = (agg_mtrs).reset_index()
    os.chdir(curdir)
    return processed

Specify the folder with `mlruns`

In [ ]:
MLRUNS_PATH = '/home/leostre/Рабочий стол/py-boost/analysis'

In [ ]:
processed = process_mlruns({}, [
    'sigmoid_5.2.1',
    'hyperbolic_5.2.1',
    'baselines_5.2.1',
    'lgbm_5.2.1',
    'xgboost_5.2.1',
    # 'sigmoid_5.2',
], DATASET, MLRUNS_PATH)

In [ ]:
import numpy as np 


def compare_prepare(modified, base, filter, method='standard', include_detailed=False):
    modified_data = processed[modified].reset_index()
    if base:
        basic = processed[base].reset_index()
    else:
        basic = modified_data.copy()
        basic.loc[:, :] = np.zeros(modified_data.shape)

    hps = set(modified_data.columns) & set(basic.columns)
    for c in ['std', 'index', 'experiment']: 
        hps.discard(c)
        if c in modified_data.columns:
            modified_data = modified_data.drop(c, axis=1)
            basic = basic.drop(c, axis=1)

    hps.discard('value'); 
    hps = list(hps)
    difference = pd.merge(modified_data, basic, on=hps, suffixes=['_m', '_b'])
    assert difference.shape[0] > 0
    metrics = difference['metric']
    difference['value'] = (difference['value_m'] - difference['value_b']) * 100
    
    if include_detailed:
        is_computational = metrics.isin(COMPUTATIONAL_METRICS).values
    else:
        is_computational = (metrics.isin(COMPUTATIONAL_METRICS) & ~metrics.isin(DETAILED_COMPUTATIONAL)).values
    difference.loc[is_computational, 'value'] = difference.loc[is_computational, 'value'] / difference.loc[is_computational, 'value_b']
    difference.drop(['value_b', 'value_m'], axis=1, inplace=True)
    for c in TO_FLOAT:
        if not c in difference:
            continue
        difference[c] = difference[c].astype(float)
    for c in TO_INT:
        if not c in difference:
            continue
        difference[c] = difference[c].astype(int)
    difference = filter_df(difference, filter)
    difference = difference.set_index(hps).sort_index().reset_index()
    # difference.sort_values(hps, inplace=True)
    return difference


In [ ]:
all_data = pd.concat([df.reset_index() for df in
    processed.values()
], axis=0)

hamming_loss = all_data[(all_data.metric == 'accuracy')].copy() 
hamming_loss['value'] = 1 - hamming_loss['value']
hamming_loss['metric'] = 'hamming_loss'

all_data = pd.concat(
    [all_data, hamming_loss], axis=0
)

## Main comparison

In case, we wanna get the estimation of relative chenges between two experiments we use `compare_prepare`

In [ ]:
d = compare_prepare('sigmoid_5.2.1', 'baselines_5.2.1', {
#     'dataset': 'cifar10', 
                                                       #  'sketch_method': 'topk', 
                                                        # 'lr': 0.1, 
                                                   #   'smoothing_alpha': 0.9, 
                                                    # 'stabilization_threshold': 1.0
                                                     })

Main interactive wisualization:



In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from ipywidgets import interact, widgets, VBox, HBox, Output
from IPython.display import display, clear_output


def create_advanced_interactive_scatter_with_baseline(df, x_metric=None, y_metric=None, signature_flag=False):
    """
    Enhanced version with baseline comparison - with flexible hyperparameter matching.
    Can automatically handle NaN values or manually select matching columns.
    Now with dynamic metric selection widgets.
    
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe in long format
    x_metric, y_metric : str, optional
        Initial metrics to plot (if None, will use first two available metrics)
    signature_flag : bool
        Add zero lines or not
    """
    
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Get all columns for selection
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    
    numerical_cols = [col for col in id_vars if df[col].dtype in ['float64', 'int64'] and col not in categorical_cols]
    all_metrics = [col for col in df_pivoted.columns if col not in id_vars]
    
    # Set default metrics if not provided
    if x_metric is None and len(all_metrics) > 0:
        x_metric = all_metrics[0]
    if y_metric is None and len(all_metrics) > 1:
        y_metric = all_metrics[1]
    elif y_metric is None and len(all_metrics) > 0:
        y_metric = all_metrics[0]
    
    # Get unique experiments
    all_experiments = sorted(df_pivoted['experiment'].unique())
    
    # Create widgets
    filter_widgets = {}
    for col in categorical_cols:
        unique_vals = sorted(df_pivoted[col].dropna().unique())
        filter_widgets[col] = widgets.SelectMultiple(
            options=unique_vals,
            description=col[:15],
            layout=widgets.Layout(width='250px'),
            style={'description_width': 'initial'}
        )
    
    # Metric selection widgets
    x_metric_widget = widgets.Dropdown(
        options=all_metrics,
        value=x_metric,
        description='X-axis metric:',
        layout=widgets.Layout(width='250px'),
        style={'description_width': 'initial'}
    )
    
    y_metric_widget = widgets.Dropdown(
        options=all_metrics,
        value=y_metric,
        description='Y-axis metric:',
        layout=widgets.Layout(width='250px'),
        style={'description_width': 'initial'}
    )
    
    # Swap axes button
    swap_axes_button = widgets.Button(
        description='⇄ Swap Axes',
        layout=widgets.Layout(width='120px'),
        button_style='primary'
    )
    
    def swap_axes(b):
        current_x = x_metric_widget.value
        current_y = y_metric_widget.value
        x_metric_widget.value = current_y
        y_metric_widget.value = current_x
    
    swap_axes_button.on_click(swap_axes)
    
    # Baseline selection widget
    baseline_widget = widgets.Dropdown(
        options=['None'] + all_experiments,
        value='None',
        description='Baseline experiment:',
        layout=widgets.Layout(width='250px'),
        style={'description_width': 'initial'}
    )
    
    # Matching strategy selection
    matching_strategy_widget = widgets.RadioButtons(
        options=['Auto (use non-NaN columns)', 'Manual selection'],
        value='Auto (use non-NaN columns)',
        description='Matching strategy:',
        layout=widgets.Layout(width='300px'),
        style={'description_width': 'initial'}
    )
    
    # Manual column selection for matching
    all_hp_cols = categorical_cols + numerical_cols
    matching_cols_widget = widgets.SelectMultiple(
        options=all_hp_cols,
        value=all_hp_cols[:min(5, len(all_hp_cols))],  # Default to first 5 columns
        description='Match on:',
        layout=widgets.Layout(width='300px', height='150px'),
        style={'description_width': 'initial'},
        disabled=True  # Initially disabled
    )
    
    # Enable/disable manual selection based on strategy
    def update_matching_cols_enabled(change):
        matching_cols_widget.disabled = (change['new'] != 'Manual selection')
    matching_strategy_widget.observe(update_matching_cols_enabled, names='value')
    
    # Connect lines toggle
    connect_lines_widget = widgets.Checkbox(
        value=True,
        description='Connect to baseline',
        layout=widgets.Layout(width='200px')
    )
    
    # Show baseline only toggle
    show_baseline_only_widget = widgets.Checkbox(
        value=False,
        description='Show only baseline-connected points',
        layout=widgets.Layout(width='250px')
    )
    
    # Show NaN points toggle
    show_nan_widget = widgets.Checkbox(
        value=True,
        description='Show points with NaN values',
        layout=widgets.Layout(width='250px')
    )
    
    # Color by widget
    color_options = ['None', 'Auto (All columns)'] + categorical_cols + numerical_cols + all_metrics + ['experiment']
    color_by_widget = widgets.Dropdown(
        options=color_options,
        value='experiment',
        description='Color by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Size by widget
    size_by_widget = widgets.Dropdown(
        options=['None'] + numerical_cols + all_metrics,
        value='None',
        description='Size by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Opacity slider
    opacity_slider = widgets.FloatSlider(
        value=0.7,
        min=0.1,
        max=1.0,
        step=0.05,
        description='Opacity:',
        layout=widgets.Layout(width='250px')
    )
    
    # Point size slider
    point_size_slider = widgets.IntSlider(
        value=8,
        min=2,
        max=20,
        description='Point size:',
        layout=widgets.Layout(width='250px')
    )
    
    # Baseline point style
    baseline_marker_widget = widgets.Dropdown(
        options=['circle', 'square', 'diamond', 'cross', 'x', 'star'],
        value='star',
        description='Baseline marker:',
        layout=widgets.Layout(width='250px')
    )
    
    # Baseline point size
    baseline_size_widget = widgets.IntSlider(
        value=12,
        min=5,
        max=25,
        description='Baseline size:',
        layout=widgets.Layout(width='250px')
    )
    
    # NaN point style
    nan_marker_widget = widgets.Dropdown(
        options=['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'triangle-down'],
        value='diamond',
        description='NaN marker:',
        layout=widgets.Layout(width='250px')
    )
    
    # Legend toggle
    legend_toggle = widgets.Checkbox(
        value=True,
        description='Show legend',
        layout=widgets.Layout(width='150px')
    )
    
    # Grid toggle
    grid_toggle = widgets.Checkbox(
        value=True,
        description='Show grid',
        layout=widgets.Layout(width='150px')
    )
    
    # Color scale for numerical data
    color_scale_widget = widgets.Dropdown(
        options=['Viridis', 'Plasma', 'Inferno', 'Magma', 'Cividis', 'Blues', 'Reds', 'Greens'],
        value='Viridis',
        description='Color scale:',
        layout=widgets.Layout(width='250px')
    )
    
    output_widget = Output()
    
    def get_matching_columns(df_filtered, baseline_name):
        """Determine which columns to use for matching based on strategy"""
        if matching_strategy_widget.value == 'Manual selection':
            # Use manually selected columns
            matching_cols = list(matching_cols_widget.value)
            return matching_cols
        
        # Auto strategy: use columns that don't have NaN in either baseline or non-baseline
        if baseline_name == 'None' or baseline_name not in df_filtered['experiment'].values:
            return []
        
        baseline_df = df_filtered[df_filtered['experiment'] == baseline_name]
        non_baseline_df = df_filtered[df_filtered['experiment'] != baseline_name]
        
        valid_cols = []
        for col in categorical_cols + numerical_cols:
            if col in df_filtered.columns:
                # Check if column has NaN in baseline
                baseline_has_nan = baseline_df[col].isna().any()
                # Check if column has NaN in non-baseline
                non_baseline_has_nan = non_baseline_df[col].isna().any()
                
                # Only use column if it has no NaN values in either dataset
                if not baseline_has_nan and not non_baseline_has_nan:
                    valid_cols.append(col)
        
        return valid_cols
    
    def find_baseline_matches(df_filtered, baseline_name):
        """Find matching baseline points using flexible column selection"""
        if baseline_name == 'None' or baseline_name not in df_filtered['experiment'].values:
            return [], []
        
        # Separate baseline and non-baseline data
        baseline_df = df_filtered[df_filtered['experiment'] == baseline_name].copy()
        non_baseline_df = df_filtered[df_filtered['experiment'] != baseline_name].copy()
        
        # Get matching columns based on strategy
        matching_cols = get_matching_columns(df_filtered, baseline_name)
        
        if not matching_cols:
            # If no matching columns found, try to use at least one column
            available_cols = [col for col in categorical_cols + numerical_cols if col in df_filtered.columns]
            if available_cols:
                matching_cols = [available_cols[0]]
                print(f"Warning: No valid matching columns found. Using '{matching_cols[0]}' for matching.")
            else:
                return [], []
        
        # Create a dictionary for quick lookup of baseline points
        baseline_dict = {}
        for idx, row in baseline_df.iterrows():
            # Create key from matching columns, handling NaN by converting to string
            hp_key = tuple(str(row[col]) if pd.notna(row[col]) else 'NaN' for col in matching_cols)
            baseline_dict[hp_key] = row
        
        # Find matches
        matches = []
        matched_baselines = []
        
        for idx, row in non_baseline_df.iterrows():
            hp_key = tuple(str(row[col]) if pd.notna(row[col]) else 'NaN' for col in matching_cols)
            if hp_key in baseline_dict:
                baseline_row = baseline_dict[hp_key]
                matches.append((row, baseline_row))
                matched_baselines.append(baseline_row)
        
        return matches, matched_baselines
    
    def update_plot(change=None):
        with output_widget:
            clear_output(wait=True)
            
            # Get current metrics
            current_x_metric = x_metric_widget.value
            current_y_metric = y_metric_widget.value
            
            if current_x_metric is None or current_y_metric is None:
                print("Please select both X and Y metrics")
                return
            
            # Apply filters
            filtered_df = df_pivoted.copy()
            for col, widget in filter_widgets.items():
                if widget.value:
                    filtered_df = filtered_df[filtered_df[col].isin(widget.value)]
            
            if len(filtered_df) == 0:
                print("No data matches the selected filters")
                return
            
            # Display matching strategy info
            baseline_name = baseline_widget.value
            if baseline_name != 'None':
                matching_cols = get_matching_columns(filtered_df, baseline_name)
                if matching_cols:
                    print(f"🔗 Matching baseline on columns: {', '.join(matching_cols)}")
                else:
                    print("⚠️ No matching columns found! Connections may not work properly.")
            
            # Filter baseline-only if requested
            if show_baseline_only_widget.value and baseline_name != 'None':
                # Find matches first
                matches, _ = find_baseline_matches(filtered_df, baseline_name)
                matched_indices = []
                for non_base, base in matches:
                    matched_indices.append(non_base.name)
                    matched_indices.append(base.name)
                filtered_df = filtered_df.loc[matched_indices].drop_duplicates()
                
                if len(filtered_df) == 0:
                    print("No matching points found with baseline")
                    return
            
            # Separate points with NaN values if needed
            show_nan = show_nan_widget.value
            nan_mask = filtered_df[current_x_metric].isna() | filtered_df[current_y_metric].isna()
            valid_df = filtered_df[~nan_mask].copy() if show_nan else filtered_df.copy()
            nan_df = filtered_df[nan_mask].copy() if show_nan else pd.DataFrame()
            
            # Create figure
            fig = go.Figure()
            
            # Add baseline connections for valid points
            connect_lines = connect_lines_widget.value
            
            if baseline_name != 'None' and connect_lines and len(valid_df) > 0:
                # Find matches for lines
                matches, _ = find_baseline_matches(valid_df, baseline_name)
                
                # Draw lines only for pairs where both points have valid values
                lines_drawn = 0
                for non_base, base in matches:
                    # Check if both points have valid (non-NaN) values for the metrics
                    if (pd.notna(base[current_x_metric]) and pd.notna(base[current_y_metric]) and 
                        pd.notna(non_base[current_x_metric]) and pd.notna(non_base[current_y_metric])):
                        fig.add_trace(go.Scatter(
                            x=[base[current_x_metric], non_base[current_x_metric]],
                            y=[base[current_y_metric], non_base[current_y_metric]],
                            mode='lines',
                            line=dict(color='gray', width=1.5, dash='dot'),
                            showlegend=False,
                            hoverinfo='none'
                        ))
                        lines_drawn += 1
                
                if lines_drawn == 0 and len(matches) > 0:
                    print(f"⚠️ Found {len(matches)} matches but no lines drawn due to NaN values in metrics")
            
            # Plot valid points (non-NaN)
            if len(valid_df) > 0:
                color_by = color_by_widget.value
                
                if color_by == 'None':
                    temp_fig = px.scatter(
                        valid_df,
                        x=current_x_metric,
                        y=current_y_metric,
                    )
                    for trace in temp_fig.data:
                        fig.add_trace(trace)
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        ),
                        selector=dict(mode='markers')
                    )
                elif color_by == 'Auto (All columns)':
                    valid_df['color_category'] = valid_df[categorical_cols].astype(str).agg(' | '.join, axis=1)
                    temp_fig = px.scatter(
                        valid_df,
                        x=current_x_metric,
                        y=current_y_metric,
                        color='color_category',
                        labels={'color_category': 'Configuration'}
                    )
                    for trace in temp_fig.data:
                        fig.add_trace(trace)
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        ),
                        selector=dict(mode='markers')
                    )
                else:
                    is_numerical = (color_by in numerical_cols or color_by in all_metrics) and valid_df[color_by].dtype in ['float64', 'int64']
                    
                    if is_numerical:
                        temp_fig = px.scatter(
                            valid_df,
                            x=current_x_metric,
                            y=current_y_metric,
                            color=color_by,
                            color_continuous_scale=color_scale_widget.value,
                            labels={color_by: color_by}
                        )
                    else:
                        temp_fig = px.scatter(
                            valid_df,
                            x=current_x_metric,
                            y=current_y_metric,
                            color=color_by,
                            color_discrete_sequence=px.colors.qualitative.Set1,
                            labels={color_by: color_by}
                        )
                    
                    for trace in temp_fig.data:
                        fig.add_trace(trace)
                    
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        ),
                        selector=dict(mode='markers')
                    )
            
            # Plot NaN points separately with distinct marker
            if len(nan_df) > 0 and show_nan:
                # Create hover text for NaN points
                nan_hover_text = []
                for _, row in nan_df.iterrows():
                    hp_text = '<br>'.join([f'{col}: {row[col]}' for col in categorical_cols if col in nan_df.columns])
                    nan_hover_text.append(
                        f"<b>⚠️ INCOMPLETE DATA</b><br>{hp_text}<br>"
                        f"{current_x_metric}: {row[current_x_metric] if pd.notna(row[current_x_metric]) else 'NaN'}<br>"
                        f"{current_y_metric}: {row[current_y_metric] if pd.notna(row[current_y_metric]) else 'NaN'}<br>"
                        f"Experiment: {row['experiment']}"
                    )
                
                fig.add_trace(go.Scatter(
                    x=nan_df[current_x_metric] if current_x_metric in nan_df.columns else [None]*len(nan_df),
                    y=nan_df[current_y_metric] if current_y_metric in nan_df.columns else [None]*len(nan_df),
                    mode='markers',
                    name='⚠️ Incomplete data (NaN)',
                    marker=dict(
                        symbol=nan_marker_widget.value,
                        size=point_size_slider.value,
                        color='gray',
                        opacity=0.5,
                        line=dict(color='darkgray', width=1)
                    ),
                    text=nan_hover_text,
                    hovertemplate='%{text}<extra></extra>'
                ))
            
            # Highlight baseline points with different marker (only valid ones)
            if baseline_name != 'None':
                baseline_points = valid_df[valid_df['experiment'] == baseline_name] if len(valid_df) > 0 else pd.DataFrame()
                if len(baseline_points) > 0:
                    # Get matching columns for hover info
                    matching_cols = get_matching_columns(valid_df, baseline_name)
                    matching_info = []
                    for _, row in baseline_points.iterrows():
                        match_str = '<br>'.join([f'{col}: {row[col]}' for col in matching_cols if col in baseline_points.columns])
                        matching_info.append(match_str)
                    
                    fig.add_trace(go.Scatter(
                        x=baseline_points[current_x_metric],
                        y=baseline_points[current_y_metric],
                        mode='markers',
                        name=f'✨ Baseline: {baseline_name}',
                        marker=dict(
                            symbol=baseline_marker_widget.value,
                            size=baseline_size_widget.value,
                            color='red',
                            line=dict(color='darkred', width=1)
                        ),
                        text=matching_info,
                        hovertemplate='<b>✨ BASELINE</b><br>%{text}<br>' +
                                    f'{current_x_metric}: %{{x:.4f}}<br>{current_y_metric}: %{{y:.4f}}<extra></extra>'
                    ))
            
            # Apply size encoding if selected (only for valid points)
            if size_by_widget.value != 'None' and len(valid_df) > 0:
                size_col = size_by_widget.value
                if size_col in valid_df.columns:
                    size_vals = valid_df[size_col].fillna(valid_df[size_col].median())
                    min_size, max_size = size_vals.min(), size_vals.max()
                    if min_size != max_size:
                        normalized_sizes = 5 + (size_vals - min_size) / (max_size - min_size) * 15
                    else:
                        normalized_sizes = [10] * len(valid_df)
                    
                    # Update only the valid point traces (skip baseline and NaN traces)
                    for trace in fig.data:
                        if trace.mode == 'markers' and 'Baseline' not in trace.name and 'Incomplete' not in trace.name:
                            trace.marker.size = normalized_sizes
                            trace.marker.sizemode = 'area'
                            trace.marker.sizeref = 2.*max(normalized_sizes)/(40**2)
            
            # Add zero lines if requested
            if signature_flag:
                fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
                fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
            
            # Update layout based on toggles
            title = f'{current_x_metric} vs {current_y_metric}'
            if baseline_name != 'None' and connect_lines:
                title += f' (connected to baseline: {baseline_name})'
            if show_nan and len(nan_df) > 0:
                title += ' ⚠️ Gray points have NaN values'
            
            fig.update_layout(
                title=title,
                plot_bgcolor='white',
                xaxis=dict(
                    title=current_x_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                yaxis=dict(
                    title=current_y_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                hovermode='closest',
                height=600,
                showlegend=legend_toggle.value
            )
            
            fig.show()
    
    # Create UI layout
    # Top panel for metric selection
    metric_panel = HBox([
        x_metric_widget,
        y_metric_widget,
        swap_axes_button
    ])
    
    # Left panel with filters
    filter_panel = VBox([
        widgets.HTML("<b>Filters:</b>"),
        *[HBox([widget]) for widget in filter_widgets.values()]
    ])
    
    # Middle panel with baseline controls
    matching_panel = VBox([
        widgets.HTML("<b>Matching Strategy:</b>"),
        matching_strategy_widget,
        widgets.HTML("<i>Select columns for matching (manual mode):</i>"),
        matching_cols_widget
    ])
    
    baseline_panel = VBox([
        widgets.HTML("<b>Baseline Settings:</b>"),
        baseline_widget,
        connect_lines_widget,
        show_baseline_only_widget,
        show_nan_widget,
        widgets.HTML("<hr>"),
        widgets.HTML("<b>Marker Styles:</b>"),
        baseline_marker_widget,
        baseline_size_widget,
        nan_marker_widget
    ])
    
    # Right panel with styling options
    style_panel = VBox([
        widgets.HTML("<b>Styling:</b>"),
        color_by_widget,
        size_by_widget,
        color_scale_widget,
        opacity_slider,
        point_size_slider,
        legend_toggle,
        grid_toggle
    ])
    
    # Main control panel with three columns
    control_panel = VBox([
        metric_panel,
        widgets.HTML("<hr>"),
        HBox([filter_panel, VBox([matching_panel, baseline_panel]), style_panel])
    ])
    
    # Add reset button
    reset_button = widgets.Button(description='Reset All', button_style='warning')
    
    def reset_all(b):
        for widget in filter_widgets.values():
            widget.value = []
        # Reset metrics to first two if available
        if len(all_metrics) > 0:
            x_metric_widget.value = all_metrics[0]
        if len(all_metrics) > 1:
            y_metric_widget.value = all_metrics[1]
        elif len(all_metrics) > 0:
            y_metric_widget.value = all_metrics[0]
        baseline_widget.value = 'None'
        matching_strategy_widget.value = 'Auto (use non-NaN columns)'
        matching_cols_widget.value = all_hp_cols[:min(5, len(all_hp_cols))]
        connect_lines_widget.value = True
        show_baseline_only_widget.value = False
        show_nan_widget.value = True
        color_by_widget.value = 'experiment'
        size_by_widget.value = 'None'
        opacity_slider.value = 0.7
        point_size_slider.value = 8
        legend_toggle.value = True
        grid_toggle.value = True
        color_scale_widget.value = 'Viridis'
        baseline_marker_widget.value = 'star'
        baseline_size_widget.value = 12
        nan_marker_widget.value = 'diamond'
    
    reset_button.on_click(reset_all)
    
    # Attach observers
    for widget in filter_widgets.values():
        widget.observe(update_plot, names='value')
    x_metric_widget.observe(update_plot, names='value')
    y_metric_widget.observe(update_plot, names='value')
    baseline_widget.observe(update_plot, names='value')
    matching_strategy_widget.observe(update_plot, names='value')
    matching_cols_widget.observe(update_plot, names='value')
    connect_lines_widget.observe(update_plot, names='value')
    show_baseline_only_widget.observe(update_plot, names='value')
    show_nan_widget.observe(update_plot, names='value')
    color_by_widget.observe(update_plot, names='value')
    size_by_widget.observe(update_plot, names='value')
    opacity_slider.observe(update_plot, names='value')
    point_size_slider.observe(update_plot, names='value')
    legend_toggle.observe(update_plot, names='value')
    grid_toggle.observe(update_plot, names='value')
    color_scale_widget.observe(update_plot, names='value')
    baseline_marker_widget.observe(update_plot, names='value')
    baseline_size_widget.observe(update_plot, names='value')
    nan_marker_widget.observe(update_plot, names='value')
    
    # Initial plot
    update_plot()
    
    # Display complete interface
    display(VBox([control_panel, reset_button, output_widget]))
    
    return (filter_widgets, x_metric_widget, y_metric_widget, baseline_widget, 
            matching_cols_widget, color_by_widget, size_by_widget)

### NB

`nan` corresponds to baselines which don't have additional hyperparameters (`smoothing alpha` for SketchBoost or `sketch_outputs` for lgbm) <br>

To set the baseline matching columns preferably manually choose combination: 'dataset' + 'lr' + 'subsample' <br>
And set 'Show only baseline-connected' True <br>
For comparison with baselines option 'nan' should be included. <br>

Most volatile huperparameters are: `stabilization_threshold` & `lr`

In [ ]:
create_advanced_interactive_scatter_with_baseline(all_data.drop(['index', 'std'], axis=1).fillna('nan'), signature_flag=False)